# Demonstration Notebook: General Use

This notebook demonstrates the generalized app API when looking at data in the Notebook setting. Additional documentation about the generalized app API can be found here: https://jdaviz.readthedocs.io/en/latest/. These generalized workflows are supported as of Jdaviz v5.0.

### PLEASE NOTE
Running cells in rapid succession (e.g., when selecting 'Run All Cells') may cause errors or other bugs in the app. If these occur, you may need to rerun the cell or otherwise restart the notebook.

In [ ]:
from time import sleep
import warnings
# Ignore warnings when loading data
warnings.simplefilter('ignore')

from astropy import units as u
from astropy.coordinates import SkyCoord, Angle
from photutils.aperture import CircularAperture, SkyCircularAperture
from regions import PixCoord, CirclePixelRegion, CircleSkyRegion
from specutils import SpectralRegion

## Launch App

We create a Jdaviz app instance.

Multiple Jdaviz app instances can be created and used simultaneously if desired. See https://jdaviz.readthedocs.io/en/latest/quickstart.html#multiple-jdaviz-instances for more details.

In [ ]:
# Calling jd will implicitly create a jdaviz app instance
import jdaviz as jd

# Another option is to use gca() which stands for 'get current app'.
# It is not strictly necessary here but can be useful
# if multiple instances of the app are instantiated.
# jd = jdaviz.gca()

## Load Data into the App

Now we load observations. If you already have the files on your local machine, you can 
load them by passing their file paths to `load` as strings. For this example, 
we will retrieve remote data from [MAST](https://mast.stsci.edu/) via `astroquery`
by passing the observation's URI as a string. By default, the downloaded files are 
saved to the current working directory. If you set `cache=False` in the `load` 
call, it will re-download the file if desired. When using `load`, it is advisable
to specify the format, in this case 'Image'.

In this example, we use two of the JWST NIRCam
observations of the Cartwheel Galaxy. These are large files (~500 MB each), so they may take
some time to download, but after the first time running this notebook they will be cached locally.
If you want to look at images from other filters, you can uncomment the lines with those filenames.
If you want an example using smaller files, see the ImvizDitheredExample notebook, which
includes the same workflow as this one, but using two HST ACS files that are much smaller
than the JWST files used here.

After downloading, each file is loaded into the app. We ignore some warnings that appear during parsing.

In [ ]:
filenames = [
    'jw02727-o002_t062_nircam_clear-f090w_i2d.fits',
    # 'jw02727-o002_t062_nircam_clear-f150w_i2d.fits',
    # 'jw02727-o002_t062_nircam_clear-f200w_i2d.fits',
    'jw02727-o002_t062_nircam_clear-f277w_i2d.fits',
    # 'jw02727-o002_t062_nircam_clear-f356w_i2d.fits',
    # 'jw02727-o002_t062_nircam_clear-f444w_i2d.fits'
]

with jd.batch_load():
    for fn in filenames:
        uri = f"mast:JWST/product/{fn}"
        jd.load(uri, format='Image', cache=True)

Alternatively, files can be loaded into the app by setting the importer attributes directly. Since this is a generalized version of Jdaviz, we can load different data types without issue. Here we load a 2D spectrum file as a collection of 1D Spectra by specifying the format as: `ldr.format = '1D Spectrum'`. This is done to illustrate the intention of the `format` attribute (or argument in the case of `jd.load()`) since a single file may be compatible with multiple supported formats. 

In [ ]:
spectrum_label = 'my_spectrum_list'
uri = f"mast:jwst/product/jw01538-o161_t002-s000000001_nirspec_f290lp-g395h-s1600a1_s2d.fits"
ldr = jd.loaders['url']
ldr.url = uri

# Specify whether to cache the data or not
ldr.cache = True
# Specify format
ldr.format = '1D Spectrum'
# Specify spectra in the list, use wildcard to grab all
# spectra at indices in the 10's
ldr.importer.extension.selected = ['1D Spectrum at index: 1?']
# Specify data label
ldr.importer.data_label = spectrum_label
# Import!
ldr.importer()

## Display App

We can now visualize the data and start off by looking at some of the basic features:

In [ ]:
# loc='sidecar' creates a panel alongside the notebook.
# When 'loc' is unspecified, the app will show beneath this cell.
jd.show(loc='sidecar')

The first thing you will probably notice is that the image in the image viewer doesn't take up the entire viewer area. 
If you press the "b" key to blink to the other loaded image, you will see that this image is 
zoomed correctly by default. The odd default zoom on the other is because the images are linked
by pixel when loaded. We can instead link by WCS (world coordinates) so that the images
will be properly aligned:

In [ ]:
# If running cells in rapid succession, this cell may fail.
# We add a slight pause before aligning the images to avoid this.
sleep(1)
jd.plugins['Orientation'].align_by = 'WCS'

## Change Viewer Settings

Panning and zooming is possible by clicking on the 
<img
  alt="pan"
  style="vertical-align:-2px;"
  width="14"
  src='data:image/svg+xml;utf8,<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 25.46 25.46" aria-hidden="true"><polygon fill="none" stroke="%23ffffff" stroke-width="1.75" stroke-linejoin="round" vector-effect="non-scaling-stroke" style="paint-order: stroke fill" points="21.45 8.78 20.03 10.19 21.57 11.73 15.21 11.73 13.67 11.73 13.67 3.83 15.21 5.36 16.62 3.95 12.67 0 8.72 3.95 10.13 5.36 11.67 3.83 11.67 11.73 10.13 11.73 3.77 11.73 5.31 10.19 3.89 8.78 -0.06 12.73 3.89 16.68 5.31 15.26 3.77 13.73 11.67 13.73 11.67 21.63 10.13 20.09 8.72 21.51 12.67 25.46 16.62 21.51 15.21 20.09 13.67 21.63 13.67 13.73 21.57 13.73 20.03 15.26 21.45 16.68 25.4 12.73 21.45 8.78" /><polygon fill="%23000" points="21.45 8.78 20.03 10.19 21.57 11.73 15.21 11.73 13.67 11.73 13.67 3.83 15.21 5.36 16.62 3.95 12.67 0 8.72 3.95 10.13 5.36 11.67 3.83 11.67 11.73 10.13 11.73 3.77 11.73 5.31 10.19 3.89 8.78 -0.06 12.73 3.89 16.68 5.31 15.26 3.77 13.73 11.67 13.73 11.67 21.63 10.13 20.09 8.72 21.51 12.67 25.46 16.62 21.51 15.21 20.09 13.67 21.63 13.67 13.73 21.57 13.73 20.03 15.26 21.45 16.68 25.4 12.73 21.45 8.78" /></svg>' />
 icon, then clicking and dragging the mouse around the image. It is also possible to zoom in and out with the scroll wheel. To change the stretch and colormap, navigate to **Plot Options** in **Settings** (<!-- ../jdaviz/data/icons/cog.svg --><img src="data:image/svg+xml;utf8,<svg%20xmlns%3D%22http://www.w3.org/2000/svg%22%20viewBox%3D%220%200%2024%2024%22><path%20d%3D%22M12%2C15.5A3.5%2C3.5%200%200%2C1%208.5%2C12A3.5%2C3.5%200%200%2C1%2012%2C8.5A3.5%2C3.5%200%200%2C1%2015.5%2C12A3.5%2C3.5%200%200%2C1%2012%2C15.5M19.43%2C12.97C19.47%2C12.65%2019.5%2C12.33%2019.5%2C12C19.5%2C11.67%2019.47%2C11.34%2019.43%2C11L21.54%2C9.37C21.73%2C9.22%2021.78%2C8.95%2021.66%2C8.73L19.66%2C5.27C19.54%2C5.05%2019.27%2C4.96%2019.05%2C5.05L16.56%2C6.05C16.04%2C5.66%2015.5%2C5.32%2014.87%2C5.07L14.5%2C2.42C14.46%2C2.18%2014.25%2C2%2014%2C2H10C9.75%2C2%209.54%2C2.18%209.5%2C2.42L9.13%2C5.07C8.5%2C5.32%207.96%2C5.66%207.44%2C6.05L4.95%2C5.05C4.73%2C4.96%204.46%2C5.05%204.34%2C5.27L2.34%2C8.73C2.21%2C8.95%202.27%2C9.22%202.46%2C9.37L4.57%2C11C4.53%2C11.34%204.5%2C11.67%204.5%2C12C4.5%2C12.33%204.53%2C12.65%204.57%2C12.97L2.46%2C14.63C2.27%2C14.78%202.21%2C15.05%202.34%2C15.27L4.34%2C18.73C4.46%2C18.95%204.73%2C19.03%204.95%2C18.95L7.44%2C17.94C7.96%2C18.34%208.5%2C18.68%209.13%2C18.93L9.5%2C21.58C9.54%2C21.82%209.75%2C22%2010%2C22H14C14.25%2C22%2014.46%2C21.82%2014.5%2C21.58L14.87%2C18.93C15.5%2C18.67%2016.04%2C18.34%2016.56%2C17.94L19.05%2C18.95C19.27%2C19.03%2019.54%2C18.95%2019.66%2C18.73L21.66%2C15.27C21.78%2C15.05%2021.73%2C14.78%2021.54%2C14.63L19.43%2C12.97Z%22%20fill%3D%22%23000000%22%20stroke%3D%22%23ffffff%22%20stroke-width%3D%221.75%22%20stroke-linecap%3D%22round%22%20stroke-linejoin%3D%22round%22%20vector-effect%3D%22non-scaling-stroke%22%20style%3D%22paint-order:%20stroke%20fill%22%20/></svg>" width="14" style="vertical-align:-2px;" />).

We can also change these programmatically, for example the stretch:

In [ ]:
image_viewer = jd.viewers['Image']
image_viewer.stretch_options

In [ ]:
image_viewer.stretch = 'sqrt'

the colormap:

In [ ]:
image_viewer.colormap_options

In [ ]:
image_viewer.set_colormap('Viridis')

the limits via the autocut using percentile option:

In [ ]:
image_viewer.autocut_options

In [ ]:
image_viewer.cuts = '95%'
image_viewer.cuts

or the limits directly:

In [ ]:
image_viewer.cuts = (0, 2)

Note also that in the above example there are mouse-over coordinates visible by default.

Viewers can also be saved to file.

In [ ]:
export_plg = jd.plugins['Export']
export_plg.viewer = 'Image'
export_plg.filename = 'my_image.png'
export_plg.export()

In [ ]:
export_plg.viewer = '1D Spectrum'
export_plg.filename = 'my_spectrum_list.png'
export_plg.export()

## Access Data

We can access information about the data by calling `jd.datasets` which returns a dictionary with the labels as keys, and `DataApi` objects as the values. We can retrieve data from the `DataApi` objects by calling `data_api_object.get_data()`.

In [ ]:
jd.datasets

In [ ]:
spatial_data_api_example = jd.datasets['jw02727-o002_t062_nircam_clear-f090w_i2d[SCI,1]']
spectral_data_api_example = jd.datasets[f'{spectrum_label}_index-10']

In [ ]:
spatial_data_api_example.get_data()

In [ ]:
spectral_data_api_example.get_data()

## Work with Subsets

It possible to make selections/regions in images and export these to Astropy regions. 

### User Task

Click on the circular selection tool (<img
  src="data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 23.53 24.66' aria-hidden='true'><path d='M14.44,7.27a7,7,0,1,0,7,7A7,7,0,0,0,14.44,7.27Zm-1,11h0A1.48,1.48,0,0,1,12,16.76h0a1.48,1.48,0,0,1,1.48-1.48h0a1.48,1.48,0,0,1,1.48,1.48h0A1.48,1.48,0,0,1,13.44,18.24Zm2.11-4.5h0a1.48,1.48,0,0,1-1.48-1.48h0a1.48,1.48,0,0,1,1.48-1.48h0A1.48,1.48,0,0,1,17,12.26h0A1.48,1.48,0,0,1,15.55,13.74Z' fill='%23000' stroke='%23fff' stroke-width='1.5' stroke-linejoin='round' stroke-linecap='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/><rect x='2.48' y='16.18' width='2.96' height='2.96' rx='1.48' fill='%23000' stroke='%23fff' stroke-width='1.5' stroke-linejoin='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/><rect x='4.48' y='3.3' width='2.96' height='2.96' rx='1.48' fill='%23000' stroke='%23fff' stroke-width='1.5' stroke-linejoin='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/><rect x='9.47' y='1.78' width='2.96' height='2.96' rx='1.48' fill='%23000' stroke='%23fff' stroke-width='2' stroke-linejoin='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill'/></svg>"
  width="16"
  style="vertical-align:-4px;"
/>), and drag and click to select an interesting region on the sky. We can then export this region with:

In [ ]:
regions = jd.plugins['Subset Tools'].get_regions()

In [ ]:
regions

It is possible to programmatically pass a `regions` shape, a `photutils` aperture shape, or a Numpy mask.

In [ ]:
c = SkyCoord('00h37m41.1171s -33d42m58.5745s')

my_aper_sky = SkyCircularAperture(c, 2 * u.arcsec)
# Note: the use of pixel-defined regions when
# aligned by WCS is a work in progress.
# my_aper_pix = CircularAperture((1714, 1380), r=2000)

# regions shape
my_reg_sky = CircleSkyRegion(c, Angle(4, u.arcsec))
# my_reg_pix = CirclePixelRegion(center=PixCoord(x=1526, y=1063), radius=1000)

my_regions = [my_aper_sky, my_reg_sky]
# my_regions += [my_aper_pix, my_reg_pix]
jd.plugins['Subset Tools'].import_region(my_regions, combination_mode='new')

The same is true of spectral subsets using the `SpectralRegion` class

In [ ]:
spectral_region = SpectralRegion(4*u.um, 4.4*u.um)
jd.plugins['Subset Tools'].import_region(spectral_region, combination_mode='new')

We can additionally retrieve subsets as masks by using the `get_data` method from the `DataApi` objects:

In [ ]:
# Refresh regions variable
regions = jd.plugins['Subset Tools'].get_regions()

# Grab subset labels for each type
for subset_label, subset_region in regions.items():
    if isinstance(subset_region, SpectralRegion):
        spectral_subset = subset_label
        break

for subset_label, subset_region in regions.items():
    # There can also be temporal subsets but for now
    # let's assume non-spectral regions are spatial
    if not isinstance(subset_region, SpectralRegion):
        spatial_subset = subset_label
        break

In [ ]:
# Get the data with the spectral subset applied as a mask
spectral_data_with_subset_mask = spectral_data_api_example.get_data(spectral_subset=spectral_subset)
# Extract the mask
spectral_subset_as_mask = spectral_data_with_subset_mask.mask
# Apply the mask to the spectral axis to produce the 'masked region'
spectral_subset_as_region = spectral_data_with_subset_mask.spectral_axis[~spectral_subset_as_mask]
spectral_subset_as_region

In [ ]:
# The process is slightly different for spatial data 
# given the underlying data type (nddata)
spatial_data_with_subset_mask = spatial_data_api_example.get_data(spatial_subset=spatial_subset)
spatial_subset_as_mask = spatial_data_with_subset_mask.mask
spatial_subset_as_region = spatial_data_with_subset_mask.data[~spatial_subset_as_mask]
spatial_subset_as_region

## Create/Manipulate Additonal Viewers

If you want a third viewer, run the following cell *or* click on the <img
  src="data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24'><path d='M17,13H13V17H11V13H7V11H11V7H13V11H17M19,3H5C3.89,3 3,3.89 3,5V19A2,2 0 0,0 5,21H19A2,2 0 0,0 21,19V5C21,3.89 20.1,3 19,3Z' fill='%23000' stroke='%23fff' stroke-width='1.75' stroke-linecap='round' stroke-linejoin='round' vector-effect='non-scaling-stroke' style='paint-order:stroke fill' /></svg>"
  width="14"
  style="vertical-align:-2px;"
/> icon and navigate to **Viewer** to proceed.

In [ ]:
viewer_3 = jd.new_viewers['1D Spectrum']
viewer_3.dataset.selected = [f'{spectrum_label}_index-10']
viewer_3_api = viewer_3()
# Assume the newly created viewer is last in the jd.viewers dictionary
viewer_3_name = list(jd.viewers.keys())[-1]

You can also programmatically control the new viewer, as follows.

In [ ]:
# Add another spectrum to the new viewer.
viewer_3_dm = jd.viewers[viewer_3_name].data_menu
viewer_3_dm.add_data(f'{spectrum_label}_index-18')

To destroy this new viewer, click the "x" that is right next to the viewer ID.